# Phase 4a: Banking77 Reasoning-Distilled SLM Evaluation Generation

This notebook runs `04_generate_eval.py` on Kaggle GPU to generate raw model outputs and logprobs for:
1. `b77_test` (3,080 test queries)
2. `amb_test` (200 held-out ambiguous queries)
3. `b77_train500` (500 memorisation check queries)

Across three modes:
- `free`: System chooses `<fast>` vs `<think>` autonomously
- `fast`: Prefilled with `<fast>`
- `think`: Prefilled with `<think>`

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate

In [ ]:
from pathlib import Path

# Check attached inputs
print("Checking Kaggle input directories:")
for p in Path("/kaggle/input").glob("**/*"):
    if p.is_file() and p.name in ["adapter_config.json", "test.csv", "train.jsonl", "ambiguous_test.jsonl"]:
        print(f"  {p}")

In [ ]:
# Run evaluation for the fine-tuned adapter across all 3 modes
!python -m scripts.04_generate_eval \
    --adapter /kaggle/input/banking77-qwen2-5-3b-qlora/qwen2.5_3b_banking77_final \
    --data-dir /kaggle/input/banking77-eval-data \
    --out-dir /kaggle/working/eval \
    --modes free fast think \
    --batch-size 16

In [ ]:
# Inspect generated files
eval_out = Path("/kaggle/working/eval")
print("Generated eval output files:")
for f in eval_out.glob("*.jsonl"):
    lines = sum(1 for _ in open(f, encoding='utf-8'))
    print(f"  {f.name}: {lines} records ({f.stat().st_size / 1e3:.1f} KB)")